# 第03课：学习机制（Mechanics of Learning）

这是 Deep Learning with PyTorch 第 03 课的完整实验，适合直接在 **Kaggle Notebook** 中运行。

任务：用线性模型把未知温标读数 `t_u` 换成摄氏度 `t_c`：

\[
\hat{t}_c = w \cdot t_u + b
\]

损失是均方误差（MSE）。三个递进实验：

1. 参数估计、手工梯度、学习率与输入归一化
2. `autograd` 自动求导，仍手工更新参数
3. `optimizer`、训练/验证划分、`no_grad` 与 `set_grad_enabled`

全程只用 CPU。Kaggle 已预装 PyTorch 和 matplotlib，一般不需要再安装。


## 环境设置


In [ ]:
%matplotlib inline

from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display
from matplotlib import pyplot as plt
import torch
import torch.optim as optim

KAGGLE_WORKING = Path("/kaggle/working")
ROOT = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", "/tmp/dlwpt-lesson03-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/dlwpt-lesson03-cache")

SEED = 20260902
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
torch.set_printoptions(edgeitems=2, linewidth=88, precision=6)

print("PyTorch version:", torch.__version__)
print("Execution device:", DEVICE)
print("CUDA used:", torch.cuda.is_available())
print("输出图片目录:", OUTPUT_DIR)


## 数据

`t_c` 是目标摄氏度，`t_u` 是未知温标读数。`t_un = 0.1 * t_u` 做输入归一化，后面会对比原始输入与归一化输入。


In [ ]:
t_c = torch.tensor(
    [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0],
    dtype=torch.float32,
    device=DEVICE,
)
t_u = torch.tensor(
    [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4],
    dtype=torch.float32,
    device=DEVICE,
)
t_un = 0.1 * t_u

print("t_c:", t_c)
print("t_u:", t_u)
print("t_un:", t_un)


## 模型、损失与手工梯度

线性模型：`w * x + b`。损失是 MSE。`grad_fn` 用链式法则手工计算 `loss` 对 `w`、`b` 的梯度。


In [ ]:
def model(inputs: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return w * inputs + b


def loss_fn(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    squared_diffs = (predictions - targets) ** 2
    return squared_diffs.mean()


def dloss_fn(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return 2 * (predictions - targets) / predictions.size(0)


def dmodel_dw(inputs: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    del w, b
    return inputs


def dmodel_db(inputs: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> float:
    del inputs, w, b
    return 1.0


def grad_fn(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    predictions: torch.Tensor,
    w: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    dloss_dtp = dloss_fn(predictions, targets)
    dloss_dw = dloss_dtp * dmodel_dw(inputs, w, b)
    dloss_db = dloss_dtp * dmodel_db(inputs, w, b)
    return torch.stack([dloss_dw.sum(), dloss_db.sum()])


## 三种训练循环


In [ ]:
def manual_training_loop(
    n_epochs: int,
    learning_rate: float,
    params: torch.Tensor,
    inputs: torch.Tensor,
    targets: torch.Tensor,
    *,
    print_params: bool = False,
) -> tuple[torch.Tensor, list[float]]:
    params = params.clone().to(DEVICE)
    history: list[float] = []
    report_epochs = {1, 2, 3, 10, 11, 99, 100, 4000, 5000}

    for epoch in range(1, n_epochs + 1):
        w, b = params
        predictions = model(inputs, w, b)
        loss = loss_fn(predictions, targets)
        grad = grad_fn(inputs, targets, predictions, w, b)
        params = params - learning_rate * grad
        history.append(loss.detach().item())

        if epoch in report_epochs:
            print(f"  Epoch {epoch:4d} | loss={loss.item():12.6f}")
            if print_params:
                print(f"             params={params} | grad={grad}")
        if not torch.isfinite(loss).all():
            print(f"  在epoch={epoch}检测到非有限loss，停止该组对照。")
            break

    return params, history


def autograd_training_loop(
    n_epochs: int,
    learning_rate: float,
    params: torch.Tensor,
    inputs: torch.Tensor,
    targets: torch.Tensor,
) -> tuple[torch.Tensor, list[float]]:
    params = params.clone().detach().to(DEVICE).requires_grad_()
    history: list[float] = []

    for epoch in range(1, n_epochs + 1):
        if params.grad is not None:
            params.grad.zero_()

        predictions = model(inputs, *params)
        loss = loss_fn(predictions, targets)
        loss.backward()

        with torch.no_grad():
            params -= learning_rate * params.grad

        history.append(loss.detach().item())
        if epoch % 500 == 0:
            print(f"  Epoch {epoch:4d} | loss={loss.item():10.6f}")

    return params.detach(), history


def optimizer_training_loop(
    n_epochs: int,
    optimizer: optim.Optimizer,
    params: torch.Tensor,
    inputs: torch.Tensor,
    targets: torch.Tensor,
) -> tuple[torch.Tensor, list[float]]:
    history: list[float] = []

    for epoch in range(1, n_epochs + 1):
        predictions = model(inputs, *params)
        loss = loss_fn(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.detach().item())
        if epoch % 500 == 0:
            print(f"  Epoch {epoch:4d} | loss={loss.item():10.6f}")

    return params.detach().clone(), history


def train_validation_loop(
    n_epochs: int,
    optimizer: optim.Optimizer,
    params: torch.Tensor,
    train_inputs: torch.Tensor,
    val_inputs: torch.Tensor,
    train_targets: torch.Tensor,
    val_targets: torch.Tensor,
    *,
    use_no_grad: bool,
) -> tuple[torch.Tensor, list[float], list[float]]:
    train_history: list[float] = []
    val_history: list[float] = []

    for epoch in range(1, n_epochs + 1):
        train_predictions = model(train_inputs, *params)
        train_loss = loss_fn(train_predictions, train_targets)

        if use_no_grad:
            with torch.no_grad():
                val_predictions = model(val_inputs, *params)
                val_loss = loss_fn(val_predictions, val_targets)
                assert val_loss.requires_grad is False
        else:
            val_predictions = model(val_inputs, *params)
            val_loss = loss_fn(val_predictions, val_targets)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        train_history.append(train_loss.detach().item())
        val_history.append(val_loss.detach().item())
        if epoch <= 3 or epoch % 500 == 0:
            print(
                f"  Epoch {epoch:4d} | training loss={train_loss.item():10.6f}"
                f" | validation loss={val_loss.item():10.6f}"
            )

    return params.detach().clone(), train_history, val_history


## 画图工具


In [ ]:
def show_and_save(fig, filename: str) -> None:
    path = OUTPUT_DIR / filename
    fig.tight_layout()
    fig.savefig(path)
    display(fig)
    plt.close(fig)
    print("已保存:", path)


def save_data_plot() -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=160)
    ax.set_xlabel("Unknown measurement")
    ax.set_ylabel("Target temperature (°C)")
    ax.scatter(t_u.numpy(), t_c.numpy(), label="observations")
    ax.grid(alpha=0.25)
    ax.legend()
    show_and_save(fig, "01_temperature_data.png")


def save_fit_plot(params: torch.Tensor) -> None:
    predictions = model(t_un, *params)
    order = torch.argsort(t_u)
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=160)
    ax.set_xlabel("Unknown measurement")
    ax.set_ylabel("Target temperature (°C)")
    ax.plot(
        t_u[order].numpy(),
        predictions[order].detach().numpy(),
        label="fitted linear model",
    )
    ax.scatter(t_u.numpy(), t_c.numpy(), label="observations")
    ax.grid(alpha=0.25)
    ax.legend()
    show_and_save(fig, "02_manual_linear_fit.png")


def save_learning_rate_plot(
    divergent_history: list[float],
    slow_history: list[float],
    normalized_history: list[float],
) -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=160)
    for values, label in (
        (divergent_history, "raw input, lr=1e-2"),
        (slow_history, "raw input, lr=1e-4"),
        (normalized_history, "normalized input, lr=1e-2"),
    ):
        finite_values = [value for value in values if torch.isfinite(torch.tensor(value))]
        if finite_values:
            ax.plot(range(1, len(finite_values) + 1), finite_values, label=label)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)
    ax.legend()
    show_and_save(fig, "03_learning_rate_comparison.png")


def save_train_validation_plot(train_history: list[float], val_history: list[float]) -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=160)
    ax.plot(train_history, label="training loss")
    ax.plot(val_history, label="validation loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)
    ax.legend()
    show_and_save(fig, "04_train_validation_loss.png")


## 实验1：参数估计、loss 与手工梯度

先看初始预测和 MSE，再对比有限差分与解析梯度。三组对照说明学习率和输入尺度：

- 原始输入 + `lr=1e-2`：会发散
- 原始输入 + `lr=1e-4`：稳定但慢
- 归一化输入 + `lr=1e-2`：稳定且更快


In [ ]:
print("=== 实验1：parameter estimation、loss与手工gradient ===")
w = torch.ones((), device=DEVICE)
b = torch.zeros((), device=DEVICE)
predictions = model(t_u, w, b)
initial_loss = loss_fn(predictions, t_c)
print("初始predictions:", predictions)
print(f"初始MSE: {initial_loss.item():.6f}")

x = torch.ones((), device=DEVICE)
y = torch.ones(3, 1, device=DEVICE)
z = torch.ones(1, 3, device=DEVICE)
a = torch.ones(2, 1, 1, device=DEVICE)
print(f"broadcasting shapes: x={x.shape}, y={y.shape}, z={z.shape}, a={a.shape}")
print("x * y shape:", (x * y).shape)
print("y * z shape:", (y * z).shape)
print("y * z * a shape:", (y * z * a).shape)

delta = 0.1
loss_rate_of_change_w = (
    loss_fn(model(t_u, w + delta, b), t_c)
    - loss_fn(model(t_u, w - delta, b), t_c)
) / (2.0 * delta)
learning_rate = 1e-2
w = w - learning_rate * loss_rate_of_change_w
loss_rate_of_change_b = (
    loss_fn(model(t_u, w, b + delta), t_c)
    - loss_fn(model(t_u, w, b - delta), t_c)
) / (2.0 * delta)
b = b - learning_rate * loss_rate_of_change_b
print(
    "有限差分：",
    f"dL/dw={loss_rate_of_change_w.item():.6f},",
    f"dL/db={loss_rate_of_change_b.item():.6f},",
    f"updated w={w.item():.6f}, b={b.item():.6f}",
)

analytic_grad = grad_fn(t_u, t_c, model(t_u, w, b), w, b)
print("解析gradient:", analytic_grad)

print("\n[对照A] raw input + learning_rate=1e-2：预期发散")
_, divergent_history = manual_training_loop(
    100, 1e-2, torch.tensor([1.0, 0.0]), t_u, t_c
)

print("\n[对照B] raw input + learning_rate=1e-4：稳定但较慢")
_, slow_history = manual_training_loop(
    100, 1e-4, torch.tensor([1.0, 0.0]), t_u, t_c
)

print("\n[对照C] normalized input + learning_rate=1e-2：稳定且更快")
_, normalized_history = manual_training_loop(
    100, 1e-2, torch.tensor([1.0, 0.0]), t_un, t_c
)

print("\n[完整拟合] normalized input，运行5000 epochs")
fitted_params, _ = manual_training_loop(
    5000,
    1e-2,
    torch.tensor([1.0, 0.0]),
    t_un,
    t_c,
    print_params=True,
)
final_loss = loss_fn(model(t_un, *fitted_params), t_c)
print(f"手工gradient最终params={fitted_params}, loss={final_loss.item():.6f}")

save_data_plot()
save_fit_plot(fitted_params)
save_learning_rate_plot(divergent_history, slow_history, normalized_history)
manual_params = fitted_params


## 实验2：autograd 与手工参数更新

不再手写导数，改用 `loss.backward()`。更新仍是：

```python
params -= learning_rate * params.grad
```

注意：反向传播前要把 `grad` 清零，否则梯度会累加。


In [ ]:
print("=== 实验2：autograd与手工parameter update ===")
params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
print("backward前params.grad is None:", params.grad is None)
loss = loss_fn(model(t_u, *params), t_c)
loss.backward()
print("第一次backward后的params.grad:", params.grad)
if params.grad is not None:
    params.grad.zero_()
print("zero_后的params.grad:", params.grad)

fitted_params, _ = autograd_training_loop(
    5000,
    1e-2,
    torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True),
    t_un,
    t_c,
)
final_loss = loss_fn(model(t_un, *fitted_params), t_c)
print(f"autograd最终params={fitted_params}, loss={final_loss.item():.6f}")
autograd_params = fitted_params


## 实验3：optimizer、训练/验证与梯度开关

- `SGD` 配归一化输入；`Adam` 可直接配原始输入
- 约 80/20 划分独立训练/验证样本
- 对照验证前向是否建计算图
- 用 `torch.set_grad_enabled(is_train)` 统一训练和验证前向


In [ ]:
print("=== 实验3：optimizer、training/validation与gradient开关 ===")
optimizer_names = sorted(
    name
    for name in dir(optim)
    if name[:1].isupper() and isinstance(getattr(optim, name), type)
)
print("torch.optim中可见的optimizer类（节选）:", optimizer_names[:12])

params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.SGD([params], lr=1e-5)
loss = loss_fn(model(t_u, *params), t_c)
loss.backward()
optimizer.step()
print("raw input单次SGD step后的params:", params.detach())

params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.SGD([params], lr=1e-2)
loss = loss_fn(model(t_un, *params), t_c)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print("normalized input单次标准SGD step后的params:", params.detach())

print("\n[SGD完整拟合] normalized input，5000 epochs")
params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.SGD([params], lr=1e-2)
sgd_params, _ = optimizer_training_loop(5000, optimizer, params, t_un, t_c)
print("SGD最终params:", sgd_params)

print("\n[Adam完整拟合] raw input，2000 epochs")
params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.Adam([params], lr=1e-1)
adam_params, _ = optimizer_training_loop(2000, optimizer, params, t_u, t_c)
print("Adam最终params:", adam_params)

n_samples = t_u.shape[0]
n_val = int(0.2 * n_samples)
shuffled_indices = torch.randperm(n_samples, device=DEVICE)
train_indices = shuffled_indices[:-n_val]
val_indices = shuffled_indices[-n_val:]
print("train_indices:", train_indices)
print("val_indices:", val_indices)

train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]
val_t_u = t_u[val_indices]
val_t_c = t_c[val_indices]
train_t_un = 0.1 * train_t_u
val_t_un = 0.1 * val_t_u

print("\n[train/val实验A] validation forward仍建立graph")
params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.SGD([params], lr=1e-2)
_, _, _ = train_validation_loop(
    3000,
    optimizer,
    params,
    train_t_un,
    val_t_un,
    train_t_c,
    val_t_c,
    use_no_grad=False,
)

print("\n[train/val实验B] 使用torch.no_grad()执行validation forward")
params = torch.tensor([1.0, 0.0], device=DEVICE, requires_grad=True)
optimizer = optim.SGD([params], lr=1e-2)
final_params, train_history, val_history = train_validation_loop(
    3000,
    optimizer,
    params,
    train_t_un,
    val_t_un,
    train_t_c,
    val_t_c,
    use_no_grad=True,
)
print("train/val最终params:", final_params)
save_train_validation_plot(train_history, val_history)

params_for_forward = final_params.clone().detach().requires_grad_()


def calc_forward(inputs: torch.Tensor, targets: torch.Tensor, is_train: bool) -> torch.Tensor:
    with torch.set_grad_enabled(is_train):
        predictions = model(inputs, *params_for_forward)
        return loss_fn(predictions, targets)


train_forward_loss = calc_forward(train_t_un, train_t_c, is_train=True)
val_forward_loss = calc_forward(val_t_un, val_t_c, is_train=False)
print(
    "set_grad_enabled检查：",
    f"train requires_grad={train_forward_loss.requires_grad},",
    f"val requires_grad={val_forward_loss.requires_grad}",
)
assert train_forward_loss.requires_grad is True
assert val_forward_loss.requires_grad is False


## 对照：手工梯度 vs autograd


In [ ]:
max_difference = (manual_params - autograd_params).abs().max().item()
print(f"manual vs. autograd parameter max difference: {max_difference:.8f}")
assert max_difference < 1e-4
print("所有实验完成，且全过程仅使用CPU。")
print("输出图片目录:", OUTPUT_DIR)
print("生成的文件:", sorted(path.name for path in OUTPUT_DIR.glob("*.png")))
